# Discretization sensitivity

In [1]:
import random, pickle
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from ramen.Ramen import Ramen

CSV, TARGET = "tutorial_data.csv", "Long Covid"
BINS, SEEDS = (5, 10, 15), (1, 2, 3)
OUT = Path("discretization_results")
OUT.mkdir(exist_ok=True)
PREDICTORS = [c for c in pd.read_csv(CSV, nrows=0).columns if c != TARGET]


In [2]:
runs = {}
for bins in BINS:
    for seed in SEEDS:
        path = OUT / f"bins{bins}_seed{seed}.pkl"
        if path.exists():
            runs[bins, seed] = pickle.loads(path.read_bytes())
            continue
        random.seed(seed)
        np.random.seed(seed)
        ramen = Ramen(CSV, TARGET, 0, num_bins=bins)
        ramen.random_walk(num_walks=50_000, p_value=0.05, correction="no_correction")
        ramen.genetic_algorithm(num_candidates=10, reg_factor=0.01, hard_stop=100)
        runs[bins, seed] = ramen.export_ramen_as_dict()
        path.write_bytes(pickle.dumps(runs[bins, seed]))
        print(f"bins={bins}, seed={seed}: done")


Starting removing vars with too few values, vectorizing dataframe, and initializing mutual information matrix, this might take a few minutes.
Removed 0 variables because of insufficient data. If deleted too many, please adjust the min_values
done
Optimization terminated successfully.
         Current function value: 4.165488
         Iterations: 12
         Function evaluations: 13
         Gradient evaluations: 13
23
initializing scorer
done initializing scorer
making candidates
Finished making candidates in: 0.03481740003917366
starting Genetic Algorithm
-17.253373151558726
generation: 0 best: -16.712716802475285
generation: 1 best: -16.355748949162088
generation: 2 best: -16.06927899398283
generation: 3 best: -15.872833916638585
generation: 4 best: -15.693007524850099
generation: 5 best: -15.547960565608378
generation: 6 best: -15.369047584591542
generation: 7 best: -15.270922960817975
generation: 8 best: -15.06847775104546
generation: 9 best: -14.956795337641836
generation: 10 best

In [21]:

def arrival_profile(result):
    weights = {
        variable: float(result["END_VAR_ARRIVALS"].get(variable, 0.0))
        for variable in predictors
    }

    total = sum(weights.values())
    if total:
        weights = {
            variable: value / total
            for variable, value in weights.items()
        }

    vector = np.array(
        [weights[variable] for variable in predictors],
        dtype=float,
    )

    ranking = sorted(
        predictors,
        key=lambda variable: (-weights[variable], variable),
    )

    return vector, ranking

In [22]:

predictors = PREDICTORS

def jaccard(a, b):
    union = a | b
    if not union:
        return 1.0
    return len(a & b) / len(union)


def top_rw_edges(result):
    edges = pd.DataFrame(result["RW_EDGE_VISIT"])

    edges = edges.sort_values(
        ["visits", "node_1", "node_2"],
        ascending=[False, True, True]
    ).head(10)

    return set(zip(edges["node_1"], edges["node_2"]))


# 1. Prepare comparisons
pairs = []

# Default setting: different seeds
for seed_a, seed_b in combinations(SEEDS, 2):
    pairs.append(("10 vs 10", 10, seed_a, 10, seed_b))

# Different bins: same nominal seed
for bins in [5, 15]:
    for seed in SEEDS:
        pairs.append((f"10 vs {bins}", 10, seed, bins, seed))


# 2. Calculate metrics
rows = []

for comparison, bins_a, seed_a, bins_b, seed_b in pairs:
    a = runs[bins_a, seed_a]
    b = runs[bins_b, seed_b]

    # Outcome relevance
    x, ranking_a = arrival_profile(a)
    y, ranking_b = arrival_profile(b)

    top_a = set(ranking_a[:10])
    top_b = set(ranking_b[:10])

    # Active nodes in final networks
    nodes_a = {node for edge in a["FINAL_NETWORK"] for node in edge}
    nodes_b = {node for edge in b["FINAL_NETWORK"] for node in edge}

    # Top-10 directed RW edges
    rw_edges_a = top_rw_edges(a)
    rw_edges_b = top_rw_edges(b)

    rows.append({
        "comparison": comparison,
        "seed_a": seed_a,
        "seed_b": seed_b,
        "Pearson r": float(pearsonr(x, y).statistic),
        "top-10 overlap": len(top_a & top_b) / 10,
        "active-node Jaccard": jaccard(nodes_a, nodes_b),
        "top-10 edge overlap": len(rw_edges_a & rw_edges_b) / 10,
    })


# 3. Summarize and save
details = pd.DataFrame(rows)

metrics = [
    "Pearson r",
    "top-10 overlap",
    "active-node Jaccard",
    "top-10 edge overlap",
]

summary = details.groupby("comparison", sort=False)[metrics].agg(["mean", "std"])

details.to_csv(OUT / "details.csv", index=False)
summary.to_csv(OUT / "summary.csv")

display(summary.round(3))


Pearson r        top-10 overlap        active-node Jaccard         \
                mean    std           mean    std                mean    std   
comparison                                                                     
10 vs 10       0.958  0.013          0.867  0.058               0.876  0.050   
10 vs 5        0.893  0.009          0.833  0.058               0.881  0.070   
10 vs 15       0.945  0.010          0.867  0.058               0.859  0.172   

           top-10 edge overlap         
                          mean    std  
comparison                             
10 vs 10                 0.933  0.058  
10 vs 5                  0.867  0.058  
10 vs 15                 0.933  0.058